# Flow-Aware Temporal Pattern Mining for Multi-Stage Network Intrusion Detection

**MSCAD multi-stage NIDS: executable project notebook** (M.Tech research project)

Repository: <https://github.com/banothsuresh1/Mscad-multistage-nids>

This notebook runs the code that exists in the repository today, cell by cell:

| Repository file | Role | Used in |
|---|---|---|
| `preprocessing/process_mscad_73cols.py` | Preprocessing pipeline; entry point `main()` | Sections 3-6 (imported, not modified) |
| `tests/test_process_mscad_73cols.py` | End-to-end test on synthetic data | Section 7 |
| `README.md` | Project description and usage | Section 1 |

> **Scope.** The README names deep-learning models (AE, CNN-LSTM, BiLSTM) and sequential pattern mining as the
> research goal, but the repository does not contain that code yet. This notebook therefore covers the
> implemented stage, **preprocessing and label assignment**, and starts no model training.

## 1. Project introduction

### 1.1 Research objectives
* Detect **multi-stage attacks** (reconnaissance → exploitation/flooding) by mining **temporal patterns** across
  sequences of network flows, not only single flows.
* For that, the flows of each capture have to stay in chronological order, carry their connection metadata
  (IPs, ports, protocol, timestamp) for session reconstruction, and have trustworthy labels.
* **Stage implemented here:** convert the six MSCAD CICFlowMeter captures into six independent, labelled,
  chronologically sorted 73-column CSV files.

### 1.2 Dataset: MSCAD
Six PCAP captures were converted with CICFlowMeter v4.0 into flow CSVs (about 84 columns each):

| File | Scenario / step | Attack label |
|---|---|---|
| `App-01.pcap_Flow.csv` | Scenario 1, Step 2a | HTTP_DDoS |
| `App-02.pcap_Flow.csv` | Scenario 1, Step 2b | ICMP_Flood |
| `N-0.pcap_Flow.csv` | Normal baseline | Normal |
| `Scan-1.pcap_Flow.csv` | Scenario 1, Step 1 | Port_Scan |
| `W-B-01.pcap_Flow.csv` | Scenario 2, Step 1 | Web_Crwling |
| `W-B-02.pcap_Flow.csv` | Scenario 2, Step 2 | Brute_Force |

The original **Kaggle `MSCAD.csv`** (128,799 rows × 67 columns: 66 features + `Label`) is used only as a
**read-only ground-truth lookup**. Kaggle class counts: Brute_Force 88,502 · Normal 28,502 · Port_Scan 11,081 ·
HTTP_DDoS 641 · ICMP_Flood 45 · Web_Crwling 28.

### 1.3 Repository structure
```
Mscad-multistage-nids\
├── MSCAD_Multistage_NIDS.ipynb      <- this notebook
├── README.md
├── preprocessing\
│   └── process_mscad_73cols.py      <- pipeline (entry point: main)
└── tests\
    └── test_process_mscad_73cols.py <- synthetic end-to-end test
```

### 1.4 Methodology implemented in `process_mscad_73cols.py`
Each capture is processed **independently**. The six files are never merged.
1. Load the CSV and normalize headers (`strip()`, `strip("'")`). A duplicate column such as `Fwd Header Len.1`
   is dropped only if its values are verified identical.
2. Validate the raw schema. Drop repeated header lines, which are not flows.
3. Parse the 66 features. `Infinity`/NaN/invalid values become NaN and are counted.
4. **Labelling, single-class captures (authoritative capture rule):** N-0 → Normal (21), App-01 → HTTP_DDoS (641),
   App-02 → ICMP_Flood (45), Scan-1 → Port_Scan (11,081), W-B-01 → Web_Crwling (28). Signature matches are only
   diagnostics here: a signature the reference also links to another class never changes these labels.
5. **Labelling, W-B-02 (expected Brute_Force 88,526):** every flow must match Kaggle reference rows exactly, on
   all 66 features or on 12 signature features, with exactly one permitted label. A flow with no reference match gets
   Brute_Force only if it belongs to a reference-verified attack session (same Src IP, Dst IP, Dst Port, Protocol,
   inside its time window). Anything else stops W-B-02 and is listed; nothing is guessed.
   A malformed input line that lacks exactly its leading Flow ID field is realigned and listed in `input_repairs.csv`
   (a safety net; the verified re-extracted CSVs contain none).
6. **Stage 3:** impute invalid feature values (default 0.0; `nan` keeps them as NaN).
7. **Stage 4:** stable chronological sort on `Timestamp` (`%d/%m/%Y %H:%M:%S`). The original strings are kept.
8. Enforce the 73-column schema: 6 metadata + 66 features + `Label`.
9. Write outputs, per-row label provenance and reports, then reload every output and validate it.
   Known CICFlowMeter artefacts (misread ARP frames, TCP FIN-tail flows, protocol-0 rows) are flagged in
   `_provenance/` (`Extraction_Artefact`) and counted in `extraction_artefacts.csv`; rows are never removed.

### 1.5 Expected local layout
```
D:\mscad\
├── MSCAD.csv                     <- Kaggle reference (any sub-folder of D:\mscad is searched)
├── CSV\                          <- GUI extraction (W-B-01 damaged; kept, not used)
├── CSV_REEXTRACTED\              <- the six *.pcap_Flow.csv inputs (cfm.bat, Java 8; Stage C verified)
├── CSV_73Cols\                   <- created in Section 6
└── Mscad-multistage-nids\        <- this repository (start Jupyter here)
```

### 1.6 What each cell needs
* **[no data]** runs anywhere.
* **[needs captures]** needs the six CSVs in `D:\mscad\CSV_REEXTRACTED`.
* **[needs reference]** needs the Kaggle `MSCAD.csv`.

If a cell's data is missing, it prints `[SKIPPED]` with the reason instead of crashing, and the reason is
shown in the final summary (Section 8). Run the cells **in order** (Kernel → Restart & Run All also works).

## 2. Environment setup
Imports, versions, dependency check, project root, module import and path configuration.
Nothing is installed or upgraded by this notebook.

In [ ]:
# [no data] 2.1 Standard library imports and interpreter information
import importlib.util
import logging
import os
import platform
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version
from pathlib import Path

from IPython.display import Markdown, display

print(f"Python      : {platform.python_version()} ({platform.python_implementation()})")
print(f"Executable  : {sys.executable}")
print(f"Platform    : {platform.platform()}")
print(f"Working dir : {Path.cwd()}")
if sys.version_info < (3, 9):
    print("WARNING: the repository README asks for Python 3.9 or newer.")

In [ ]:
# [no data] 2.2 Run log and notebook state.
# STEP_LOG records what actually ran; Section 8 reports only from it and from the variables below.
STEP_LOG = {}


def record(step, status, detail=""):
    """status: DONE, SKIPPED, WARNING or FAILED. Re-running a cell overwrites its entry."""
    STEP_LOG[step] = (status, detail)
    print(f"[{status}] {step}" + (f": {detail}" if detail else ""))


# State shared between sections (reset when this cell is re-run).
INPUTS_AVAILABLE = False
REFERENCE_AVAILABLE = False
REFERENCE_PATH = None
reference = None
ref_labels = None
lookup = None
walk_ok = False
walk_final = None
INPUT_DIMENSIONS = None
PIPELINE_EXIT_CODE = None
OUTPUT_DIMENSIONS = None
VALIDATION_SUMMARY = None
TEST_RESULT = None
print("Run log initialised.")

In [ ]:
# [no data] 2.3 Dependency check. The repository has no requirements file; the pipeline imports only
# the standard library plus pandas and numpy. pytest is optional (the test file also runs with plain Python).
REQUIRED_PACKAGES = ["pandas", "numpy"]
OPTIONAL_PACKAGES = ["pytest"]

dependency_rows = []
for pkg, kind in [(p, "required") for p in REQUIRED_PACKAGES] + [(p, "optional") for p in OPTIONAL_PACKAGES]:
    try:
        ver = version(pkg)
    except PackageNotFoundError:
        ver = None
    dependency_rows.append((pkg, kind, ver or "NOT INSTALLED"))
    print(f"{pkg:<8} {kind:<9} {ver or 'NOT INSTALLED'}")

missing_required = [p for p, k, v in dependency_rows if k == "required" and v == "NOT INSTALLED"]
if missing_required:
    record("2.3 Dependencies", "FAILED", f"missing {missing_required}")
    raise ImportError(
        f"Missing required packages {missing_required}. Install them into this kernel's environment with:\n"
        f'  "{sys.executable}" -m pip install ' + " ".join(missing_required)
    )

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 200)
record("2.3 Dependencies", "DONE", f"pandas {pd.__version__}, numpy {np.__version__}")

In [ ]:
# [no data] 2.4 Locate the repository root (works if Jupyter was started in the repository or a sub-folder)
MODULE_RELPATH = Path("preprocessing") / "process_mscad_73cols.py"


def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / MODULE_RELPATH).is_file():
            return candidate
    raise FileNotFoundError(
        f"Could not find {MODULE_RELPATH} above {start}. Start Jupyter in the repository folder, "
        r"e.g. D:\mscad\Mscad-multistage-nids"
    )


PROJECT_ROOT = find_project_root(Path.cwd().resolve())
print("Project root:", PROJECT_ROOT)
record("2.4 Project root", "DONE", str(PROJECT_ROOT))

In [ ]:
# [no data] 2.5 Import the existing preprocessing module from the repository (not modified, not re-implemented)
PREPROCESSING_DIR = PROJECT_ROOT / "preprocessing"
if str(PREPROCESSING_DIR) not in sys.path:
    sys.path.insert(0, str(PREPROCESSING_DIR))

import process_mscad_73cols as pm

print(f"Imported '{pm.__name__}' from {pm.__file__}")


def configure_logging():
    """Show the module's log messages in this notebook.

    pm.main() adds a log-file handler on every call (see pm._setup_logging), so handlers left by an
    earlier run are closed here first. Closing them also releases processing_log.txt on Windows.
    """
    for handler in list(pm.LOG.handlers):
        pm.LOG.removeHandler(handler)
        handler.close()
    stream = logging.StreamHandler(sys.stdout)
    stream.setFormatter(logging.Formatter("%(asctime)s %(levelname)-7s %(message)s", "%H:%M:%S"))
    pm.LOG.addHandler(stream)
    pm.LOG.setLevel(logging.INFO)
    pm.LOG.propagate = False


configure_logging()
record("2.5 Import preprocessing module", "DONE", "process_mscad_73cols")

In [ ]:
# [no data] 2.6 Paths and options. Edit this cell if your layout differs from Section 1.5.
BASE_DIR_OVERRIDE = None        # e.g. Path(r"D:\mscad"); None = automatic (see below)
REFERENCE_OVERRIDE = None       # e.g. Path(r"D:\mscad\MSCAD.csv"); None = search like the script does
IMPUTATION = pm.DEFAULT_IMPUTATION      # "zero" (script default) or "nan"
INSPECT_FILE = "App-01.pcap_Flow.csv"   # capture used for the step-by-step walkthrough in Section 5
RUN_FULL_PIPELINE = True        # Section 6 writes outputs to <BASE_DIR>\CSV_73Cols
RUN_TEST_SUITE = True           # Section 7 runs the synthetic end-to-end test (about 30-60 s)


def resolve_base_dir():
    """Order: explicit override > MSCAD_BASE_DIR environment variable > the folder that contains the
    repository, if it has a CSV sub-folder > the script default (pm.BASE_DIR = D:\\mscad)."""
    if BASE_DIR_OVERRIDE:
        return Path(BASE_DIR_OVERRIDE), "BASE_DIR_OVERRIDE"
    if os.environ.get("MSCAD_BASE_DIR"):
        return Path(os.environ["MSCAD_BASE_DIR"]), "MSCAD_BASE_DIR environment variable"
    if (PROJECT_ROOT.parent / pm.INPUT_SUBDIR).is_dir():
        return PROJECT_ROOT.parent, "folder containing the repository"
    return pm.BASE_DIR, "script default (pm.BASE_DIR)"


assert INSPECT_FILE in pm.INPUT_FILES, f"INSPECT_FILE must be one of {list(pm.INPUT_FILES)}"
assert IMPUTATION in ("zero", "nan"), "IMPUTATION must be 'zero' or 'nan'"

BASE_DIR, BASE_DIR_SOURCE = resolve_base_dir()
INPUT_DIR = BASE_DIR / pm.INPUT_SUBDIR
OUTPUT_DIR = BASE_DIR / pm.OUTPUT_SUBDIR

display(pd.DataFrame(
    [
        ("Project root", PROJECT_ROOT, PROJECT_ROOT.is_dir()),
        (f"Base directory ({BASE_DIR_SOURCE})", BASE_DIR, BASE_DIR.is_dir()),
        ("Input directory (six captures)", INPUT_DIR, INPUT_DIR.is_dir()),
        ("Output directory (created in Section 6)", OUTPUT_DIR, OUTPUT_DIR.is_dir()),
        ("Reference override", REFERENCE_OVERRIDE or "automatic search", "-"),
        ("Imputation", IMPUTATION, "-"),
        ("Walkthrough capture", INSPECT_FILE, "-"),
    ],
    columns=["Setting", "Value", "Exists"],
))
record("2.6 Configuration", "DONE", f"BASE_DIR={BASE_DIR}")

## 3. Repository and dataset verification
Checks the repository files and the module API, and checks that the six captures and the Kaggle reference
exist and have the columns the module expects. All column names come from the module's own constants
(`pm.METADATA_COLUMNS`, `pm.FEATURE_COLUMNS`, `pm.DROP_COLUMNS`, `pm.SIGNATURE_COLUMNS`).

In [ ]:
# [no data] 3.1 Repository contents
SKIP_PARTS = {".git", "__pycache__", ".ipynb_checkpoints"}
repo_files = [
    {"File": p.relative_to(PROJECT_ROOT).as_posix(), "Size (KB)": round(p.stat().st_size / 1024, 1)}
    for p in sorted(PROJECT_ROOT.rglob("*"))
    if p.is_file() and not (set(p.relative_to(PROJECT_ROOT).parts) & SKIP_PARTS)
]
display(pd.DataFrame(repo_files))

dependency_files = [n for n in ("requirements.txt", "pyproject.toml", "environment.yml", "setup.py")
                    if (PROJECT_ROOT / n).exists()]
print("Dependency files:", dependency_files or "none (pipeline needs only pandas and numpy; see 2.3)")
TEST_FILE = PROJECT_ROOT / "tests" / "test_process_mscad_73cols.py"
print("Test file present:", TEST_FILE.is_file())
record("3.1 Repository contents", "DONE", f"{len(repo_files)} files")

In [ ]:
# [no data] 3.2 Verify the module provides every function this notebook calls, and its schema constants
REQUIRED_API = [
    "read_flow_csv", "read_capture_csv", "normalize_columns", "resolve_duplicate_columns", "validate_raw_schema",
    "locate_reference_dataset", "validate_reference_dataset", "prepare_signature_lookup",
    "parse_feature_columns", "match_ground_truth_labels", "assign_capture_labels", "resolve_mixed_capture_labels",
    "clean_invalid_numeric_values", "sort_flows_chronologically", "enforce_final_schema", "main",
]
missing_api = [name for name in REQUIRED_API if not callable(getattr(pm, name, None))]
if missing_api:
    record("3.2 Module verification", "FAILED", f"missing functions {missing_api}")
    raise AttributeError(f"process_mscad_73cols is missing: {missing_api}")

schema_checks = {
    "6 metadata columns": len(pm.METADATA_COLUMNS) == 6,
    "66 feature columns": len(pm.FEATURE_COLUMNS) == 66,
    "73 final columns, no duplicates": len(pm.FINAL_COLUMNS) == 73 == len(set(pm.FINAL_COLUMNS)),
    "Label is the final column": pm.FINAL_COLUMNS[-1] == pm.LABEL_COLUMN,
    "12 signature columns, all features": len(pm.SIGNATURE_COLUMNS) == 12 and set(pm.SIGNATURE_COLUMNS) <= set(pm.FEATURE_COLUMNS),
    "11 columns dropped": len(pm.DROP_COLUMNS) == 11,
}
display(pd.Series(schema_checks, name="OK").to_frame())

display(pd.DataFrame(
    [(f, a, ", ".join(sorted(pm.ALLOWED_LABELS[f])), f in pm.BASELINE_FILES) for f, a in pm.INPUT_FILES.items()],
    columns=["Capture", "Attack label", "Permitted labels", "Baseline"],
))
display(pd.DataFrame({"Class ID": pm.LABEL_ENCODING, "Kaggle count": pm.EXPECTED_CLASS_COUNTS}))
print("Victim subnet:", pm.VICTIM_NETWORK, "| Timestamp format:", pm.TIMESTAMP_FORMAT)

if all(schema_checks.values()):
    record("3.2 Module verification", "DONE", f"{len(REQUIRED_API)} functions, schema constants consistent")
else:
    record("3.2 Module verification", "WARNING", "schema constants inconsistent: see table")

In [ ]:
# [needs captures] 3.3 Check that the six CICFlowMeter CSVs exist
input_status = pd.DataFrame([
    {
        "Capture": fname,
        "Attack label": attack,
        "Exists": (INPUT_DIR / fname).is_file(),
        "Size (MB)": round((INPUT_DIR / fname).stat().st_size / 1e6, 3) if (INPUT_DIR / fname).is_file() else None,
    }
    for fname, attack in pm.INPUT_FILES.items()
])
display(input_status)

INPUTS_AVAILABLE = bool(input_status["Exists"].all())
if INPUTS_AVAILABLE:
    record("3.3 Input captures", "DONE", f"all six found in {INPUT_DIR}")
else:
    missing = input_status.loc[~input_status["Exists"], "Capture"].tolist()
    record("3.3 Input captures", "SKIPPED", f"missing in {INPUT_DIR}: {missing}. Cells marked [needs captures] will skip")

In [ ]:
# [needs reference] 3.4 Locate the Kaggle MSCAD.csv with the module's own search
# (same search roots and exclusions as pm.main(): BASE_DIR, plus sibling folders whose name contains "mscad")
REFERENCE_PATH = None
try:
    if REFERENCE_OVERRIDE is not None:
        REFERENCE_PATH = Path(REFERENCE_OVERRIDE)
        if not REFERENCE_PATH.is_file():
            raise pm.ReferenceDatasetError(f"REFERENCE_OVERRIDE not found: {REFERENCE_PATH}")
    else:
        search_roots = [BASE_DIR]
        if BASE_DIR.parent.is_dir():
            search_roots += [d for d in sorted(BASE_DIR.parent.iterdir())
                             if d.is_dir() and "mscad" in d.name.lower() and d.resolve() != BASE_DIR.resolve()]
        print("Search roots:", [str(r) for r in search_roots])
        REFERENCE_PATH, examined = pm.locate_reference_dataset(search_roots, [INPUT_DIR, OUTPUT_DIR])
        display(pd.DataFrame(examined))
    record("3.4 Kaggle reference", "DONE", str(REFERENCE_PATH))
except pm.ReferenceDatasetError as exc:
    REFERENCE_PATH = None
    record("3.4 Kaggle reference", "SKIPPED", f"{str(exc).rstrip('.')}. Set REFERENCE_OVERRIDE in 2.6 if the file is elsewhere")

REFERENCE_AVAILABLE = REFERENCE_PATH is not None

In [ ]:
# [needs captures] [needs reference] 3.5 Required columns (header only; column values are checked in Sections 5-6)
if INPUTS_AVAILABLE:
    header_rows = []
    for fname in pm.INPUT_FILES:
        try:
            columns = list(pm.normalize_columns(pd.read_csv(INPUT_DIR / fname, nrows=0, encoding="utf-8-sig")).columns)
            report = pm.validate_raw_schema(columns, fname)
            status = "OK"
        except pm.SchemaError as exc:
            report, status = {}, f"FAILED: {exc}"
        except Exception as exc:  # unreadable header
            columns, report, status = [], {}, f"FAILED: {type(exc).__name__}: {exc}"
        header_rows.append({
            "Capture": fname,
            "Columns": len(columns),
            "Status": status,
            "Missing drop columns": ", ".join(report.get("missing_drop_columns", [])),
            "Unexpected columns": ", ".join(report.get("unexpected_columns", [])),
            "Has raw Label": report.get("has_raw_label"),
        })
    header_check = pd.DataFrame(header_rows)
    display(header_check)
    if (header_check["Status"] == "OK").all():
        record("3.5a Capture columns", "DONE", "all required metadata and feature columns present")
    else:
        record("3.5a Capture columns", "WARNING", "see table: these files will fail in Section 6")
    print("Note: a '<name>.1' column under 'Unexpected columns' is a duplicate; the module drops it only if its "
          "values are identical (checked in 5.3 and in Section 6).")
else:
    record("3.5a Capture columns", "SKIPPED", "captures not available")

if REFERENCE_AVAILABLE:
    ref_columns = list(pm.normalize_columns(pd.read_csv(REFERENCE_PATH, nrows=0, encoding="utf-8-sig")).columns)
    expected_ref = pm.FEATURE_COLUMNS + [pm.LABEL_COLUMN]
    ref_missing = [c for c in expected_ref if c not in ref_columns]
    ref_extra = [c for c in ref_columns if c not in expected_ref]
    print(f"Reference: {len(ref_columns)} columns | missing {ref_missing or 'none'} | extra {ref_extra or 'none'} "
          f"| same order as schema: {ref_columns == expected_ref}")
    record("3.5b Reference columns", "DONE" if not (ref_missing or ref_extra) else "WARNING",
           f"{len(ref_columns)} columns")
else:
    record("3.5b Reference columns", "SKIPPED", "reference not available")

## 4. Data loading
Data is loaded with the repository's loader `pm.read_flow_csv()` followed by `pm.normalize_columns()`, as
`pm.main()` does. Text columns (IPs, ports, protocol, timestamp, Flow ID, Label) are read as strings, and
features are parsed with round-trip float precision. Files are only read, never written.

*Missing values* below means NaN after loading. `Infinity` values are not NaN; they are counted in 5.6.

In [ ]:
# [no data] 4.1 Display helper (presentation only, no preprocessing)
def describe_frame(df, title, n_head=5):
    display(Markdown(f"**{title}**: {df.shape[0]:,} rows × {df.shape[1]} columns"))
    print("Columns:", list(df.columns))
    display(df.head(n_head))
    display(Markdown("Data types (count of columns per dtype, then per column):"))
    display(df.dtypes.astype(str).value_counts().rename_axis("dtype").to_frame("columns"))
    display(df.dtypes.astype(str).to_frame("dtype").T)
    missing = df.isna().sum()
    missing = missing[missing > 0]
    if missing.empty:
        print("Missing values (NaN): none")
    else:
        display(Markdown("Missing values (NaN) per column:"))
        display(missing.to_frame("NaN count"))

In [ ]:
# [needs reference] 4.2 Load the Kaggle reference (read-only)
if REFERENCE_AVAILABLE:
    try:
        t0 = time.time()
        reference = pm.normalize_columns(pm.read_flow_csv(REFERENCE_PATH))
        describe_frame(reference, f"Kaggle reference {REFERENCE_PATH.name}")
        display(reference[pm.LABEL_COLUMN].value_counts().to_frame("rows"))
        record("4.2 Load reference", "DONE", f"{reference.shape[0]:,} x {reference.shape[1]} in {time.time() - t0:.1f}s")
    except Exception as exc:
        reference = None
        record("4.2 Load reference", "FAILED", f"{type(exc).__name__}: {exc}")
else:
    record("4.2 Load reference", "SKIPPED", "reference not available")

In [ ]:
# [needs captures] 4.3 Load the walkthrough capture (INSPECT_FILE) as the module reads it
if INPUTS_AVAILABLE:
    try:
        raw_inspect = pm.normalize_columns(pm.read_capture_csv(INPUT_DIR / INSPECT_FILE)[0])
        describe_frame(raw_inspect, f"Raw capture {INSPECT_FILE}")
        record("4.3 Load walkthrough capture", "DONE", f"{INSPECT_FILE}: {raw_inspect.shape[0]:,} x {raw_inspect.shape[1]}")
    except Exception as exc:
        record("4.3 Load walkthrough capture", "FAILED", f"{type(exc).__name__}: {exc}")
else:
    record("4.3 Load walkthrough capture", "SKIPPED", "captures not available")

In [ ]:
# [needs captures] 4.4 Dimensions of all six captures. Each file is loaded and released on its own; nothing is merged.
if INPUTS_AVAILABLE:
    dim_rows = []
    for fname in pm.INPUT_FILES:
        try:
            df = pm.normalize_columns(pm.read_flow_csv(INPUT_DIR / fname))
            dim_rows.append({
                "Capture": fname,
                "Rows": len(df),
                "Columns": df.shape[1],
                "Columns containing NaN": int(df.isna().any().sum()),
                "Raw CICFlowMeter Label values": dict(df[pm.LABEL_COLUMN].value_counts().head(3)) if pm.LABEL_COLUMN in df else "no Label column",
            })
            del df
        except Exception as exc:
            dim_rows.append({"Capture": fname, "Rows": None, "Columns": None, "Columns containing NaN": None,
                             "Raw CICFlowMeter Label values": f"FAILED: {type(exc).__name__}: {exc}"})
    INPUT_DIMENSIONS = pd.DataFrame(dim_rows)
    display(INPUT_DIMENSIONS)
    record("4.4 Capture dimensions", "DONE", f"{int(INPUT_DIMENSIONS['Rows'].fillna(0).sum()):,} raw rows across six separate files")
else:
    record("4.4 Capture dimensions", "SKIPPED", "captures not available")

## 5. Existing preprocessing pipeline: step-by-step walkthrough
This section calls the module's own functions, in the order `pm.main()` and `pm.process_single_file()` use
them, on **one capture** (`INSPECT_FILE`), and shows the intermediate results.

* **Nothing is written to disk here.** The authoritative outputs for all six captures are produced by
  `pm.main()` in Section 6. Cell 6.4 checks that this walkthrough gives the same result as the written file.
* Between function calls, the few glue lines (selecting columns, combining labels) follow
  `process_single_file()` and are marked as such. No preprocessing logic is re-implemented.

| Step | Module function |
|---|---|
| 5.1 Validate reference | `validate_reference_dataset` |
| 5.2 Build signature lookup | `prepare_signature_lookup` |
| 5.3 Normalize headers and verify duplicate columns | `read_flow_csv`, `normalize_columns`, `resolve_duplicate_columns` |
| 5.4 Validate raw schema | `validate_raw_schema` |
| 5.5 Remove repeated header lines | inline rule of `process_single_file` |
| 5.6 Parse features, count invalid values | `parse_feature_columns` |
| 5.7 Labelling by the verified per-capture rules | `assign_capture_labels` |
| 5.8 Label sources and collisions | (display of the 5.7 result) |
| 5.9 Stage 3: invalid-value imputation | `clean_invalid_numeric_values` |
| 5.10 Stage 4: chronological sort | `sort_flows_chronologically` |
| 5.11 Enforce 73-column schema | `enforce_final_schema` |

In [ ]:
# [needs reference] 5.1 Validate the reference: shape, columns, labels, class counts
if reference is not None:
    try:
        ref_labels, ref_info = pm.validate_reference_dataset(reference, REFERENCE_PATH)
        ref_counts = pd.DataFrame({"Reference": ref_info["class_counts"], "Expected (Kaggle)": pm.EXPECTED_CLASS_COUNTS})
        ref_counts["Difference"] = ref_counts["Reference"] - ref_counts["Expected (Kaggle)"]
        display(ref_counts)
        print("Feature order matches schema:", ref_info["feature_order_matches_schema"],
              "| Label is last column:", ref_info["label_is_last_column"])
        print("Invalid numeric values in reference:", ref_info["reference_invalid_numeric_values"] or "none")
        record("5.1 Validate reference", "DONE" if ref_info["class_counts_match_expected"] else "WARNING",
               "class counts match Kaggle" if ref_info["class_counts_match_expected"] else "class counts differ from Kaggle")
    except pm.ReferenceDatasetError as exc:
        ref_labels = None
        record("5.1 Validate reference", "FAILED", str(exc))
else:
    record("5.1 Validate reference", "SKIPPED", "reference not loaded")

In [ ]:
# [needs reference] 5.2 Build the 12-feature signature lookup (ambiguous signatures are kept and flagged, never deduplicated)
if ref_labels is not None:
    lookup, lookup_stats = pm.prepare_signature_lookup(reference, ref_labels)
    full_lookup, full_lookup_stats = pm.prepare_signature_lookup(reference, ref_labels, columns=pm.FEATURE_COLUMNS)
    print(f"66-feature exact lookup: {full_lookup_stats['distinct_signatures']:,} keys, "
          f"{full_lookup_stats['ambiguous_signatures']:,} ambiguous")
    display(pd.Series({k: v for k, v in lookup_stats.items() if not isinstance(v, dict)}, name="value").to_frame())
    display(pd.Series(lookup_stats["reference_rows_in_ambiguous_signatures_by_label"],
                      name="reference rows in ambiguous signatures").to_frame())
    ambiguous_sets = lookup.loc[lookup["__n_labels"] > 1, "__ref_labels"].value_counts()
    print("Label sets of ambiguous signatures:", dict(ambiguous_sets) or "none")
    print("Signature columns:", pm.SIGNATURE_COLUMNS)
    record("5.2 Signature lookup", "DONE",
           f"{lookup_stats['distinct_signatures']:,} signatures, {lookup_stats['ambiguous_signatures']:,} ambiguous")
else:
    lookup = None
    record("5.2 Signature lookup", "SKIPPED", "reference not validated")

In [ ]:
# [needs captures] 5.3 Read INSPECT_FILE, normalize headers and verify duplicate columns
walk_ok = False
if INPUTS_AVAILABLE and lookup is not None:
    try:
        walk_raw, walk_repairs = pm.read_capture_csv(INPUT_DIR / INSPECT_FILE)
        walk = pm.normalize_columns(walk_raw)
        walk_input_rows = len(walk)
        if walk_repairs:
            display(pd.DataFrame(walk_repairs))
        walk, walk_dup_notes = pm.resolve_duplicate_columns(walk, INSPECT_FILE)
        print(f"{INSPECT_FILE}: {walk_input_rows:,} rows, {walk.shape[1]} columns after duplicate handling")
        print("Duplicate columns:", walk_dup_notes or "none")
        walk_ok = True
        record("5.3 Headers and duplicate columns", "DONE", "; ".join(walk_dup_notes) or "no duplicates")
    except pm.SchemaError as exc:
        record("5.3 Headers and duplicate columns", "FAILED", str(exc))
else:
    record("5.3 Headers and duplicate columns", "SKIPPED", "needs the captures and the signature lookup (5.2)")

In [ ]:
# [needs captures] 5.4 Validate the raw CICFlowMeter schema before anything is dropped
if walk_ok:
    try:
        walk_schema = pm.validate_raw_schema(list(walk.columns), INSPECT_FILE)
        display(pd.Series({k: (", ".join(v) if isinstance(v, list) else v) or "none" for k, v in walk_schema.items()},
                          name="value").to_frame())
        print("Columns that will not appear in the output:",
              [c for c in walk.columns if c not in pm.FINAL_COLUMNS or c == pm.LABEL_COLUMN])
        record("5.4 Raw schema", "DONE", f"{walk_schema['raw_columns']} raw columns")
    except pm.SchemaError as exc:
        walk_ok = False
        record("5.4 Raw schema", "FAILED", str(exc))
else:
    record("5.4 Raw schema", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] 5.5 Remove repeated CSV header lines (not flows).
# Glue: same inline rule as process_single_file(); there is no separate module function for it.
if walk_ok:
    walk = walk.reset_index(drop=True)
    walk_input_row = np.arange(len(walk))
    header_lines = ((walk["Src IP"].astype(str).str.strip() == "Src IP")
                    | (walk["Timestamp"].astype(str).str.strip() == "Timestamp")).to_numpy()
    print(f"Repeated header lines: {int(header_lines.sum())} at data rows {walk_input_row[header_lines].tolist()[:20]}")
    walk = walk.loc[~header_lines].reset_index(drop=True)
    walk_input_row = walk_input_row[~header_lines]
    walk_meta = walk[pm.METADATA_COLUMNS].astype(str)
    if pm.LABEL_COLUMN in walk:
        print("Raw CICFlowMeter Label values (replaced later):", dict(walk[pm.LABEL_COLUMN].value_counts().head(5)))
    display(walk_meta.head())
    record("5.5 Repeated header lines", "DONE", f"{int(header_lines.sum())} removed, {len(walk):,} flows remain")
else:
    record("5.5 Repeated header lines", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] 5.6 Parse the 66 features; Infinity/NaN/unparseable values become NaN and are counted
if walk_ok:
    walk_parsed, walk_invalid, walk_int_cols = pm.parse_feature_columns(walk)
    invalid_table = pd.DataFrame(walk_invalid)
    invalid_table["total"] = invalid_table[["missing", "pos_inf", "neg_inf", "unparseable"]].sum(axis=1)
    invalid_table = invalid_table[invalid_table["total"] > 0]
    if invalid_table.empty:
        print("No invalid numeric values.")
    else:
        display(invalid_table)
    print(f"{len(walk_int_cols)} of 66 feature columns are integers in the input (written unchanged).")
    record("5.6 Parse features", "DONE", f"{int(invalid_table['total'].sum()) if not invalid_table.empty else 0} invalid values")
else:
    record("5.6 Parse features", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] [needs reference] 5.7 Labelling by the verified per-capture rules (pm.assign_capture_labels)
# Single-class captures: capture rule for every row. W-B-02: exact reference matching, unresolved rows stop it.
if walk_ok:
    walk_allowed = pm.ALLOWED_LABELS[INSPECT_FILE]
    mode = "single-class capture rule" if INSPECT_FILE in pm.SINGLE_CLASS_CAPTURES else "exact Kaggle reference matching"
    print(f"{INSPECT_FILE}: {mode}; permitted labels {sorted(walk_allowed)}")
    try:
        walk_labels, walk_label_prov, walk_lstats = pm.assign_capture_labels(INSPECT_FILE, walk_parsed, lookup, full_lookup, meta=walk_meta)
        display(pd.Series({k: v for k, v in walk_lstats.items() if not isinstance(v, dict)}, name="value").to_frame())
        record("5.7 Labelling", "DONE", f"{len(walk_labels):,} rows labelled ({mode})")
    except pm.UnresolvedLabelsError as exc:
        walk_ok = False
        display(exc.rows.head(20))
        record("5.7 Labelling", "FAILED", str(exc))
else:
    record("5.7 Labelling", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] 5.8 Label sources and signature collisions of the walkthrough capture
if walk_ok:
    display(pd.crosstab(pd.Series(walk_labels, name="Label"),
                        pd.Series(walk_label_prov["Label_Source"].to_numpy(), name="Label source"), margins=True))
    n_collisions = int(walk_label_prov["Signature_Collision"].sum())
    if INSPECT_FILE in pm.SINGLE_CLASS_CAPTURES:
        print(f"{n_collisions} rows have a signature the reference links only to other classes; "
              "the capture rule kept their label.")
    else:
        print(f"{n_collisions} rows share their key with reference rows of other captures' classes "
              "(verified on the permitted label).")
    expected = pm.EXPECTED_CAPTURE_DISTRIBUTION[INSPECT_FILE]
    actual = {k: int(v) for k, v in pd.Series(walk_labels).value_counts().items()}
    print(f"Distribution {actual} | verified expectation {expected} | match: {actual == expected}")
    record("5.8 Label sources", "DONE" if actual == expected else "WARNING", f"{actual}")
else:
    record("5.8 Label sources", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] 5.9 Stage 3: handle invalid feature values (IMPUTATION from 2.6)
if walk_ok:
    walk_clean, walk_modified_per_row, walk_imputed = pm.clean_invalid_numeric_values(walk_parsed, walk, walk_int_cols, IMPUTATION)
    print(f"Strategy '{IMPUTATION}': {walk_imputed} cells imputed in {int((walk_modified_per_row > 0).sum())} rows; "
          f"NaN left in features: {int(walk_clean.isna().sum().sum())}")
    display(walk_clean[pm.RATE_FEATURES].describe())
    record("5.9 Stage 3 invalid values", "DONE", f"{walk_imputed} cells imputed ({IMPUTATION})")
else:
    record("5.9 Stage 3 invalid values", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] 5.10 Stage 4: stable chronological sort; timestamp strings are kept unchanged
# Glue: metadata + cleaned features + Label are combined as in process_single_file().
if walk_ok:
    walk_frame = pd.concat([walk_meta.reset_index(drop=True), walk_clean.reset_index(drop=True)], axis=1)
    walk_frame[pm.LABEL_COLUMN] = walk_labels
    walk_frame, walk_ts = pm.sort_flows_chronologically(walk_frame)
    display(pd.Series(walk_ts, name="value").to_frame())
    display(walk_frame[pm.METADATA_COLUMNS + [pm.LABEL_COLUMN]].head(10))
    record("5.10 Stage 4 chronological sort", "DONE",
           f"{walk_ts['timestamp_min']} → {walk_ts['timestamp_max']}, {walk_ts['invalid_timestamps']} invalid")
else:
    record("5.10 Stage 4 chronological sort", "SKIPPED", "previous step did not run")

In [ ]:
# [needs captures] 5.11 Enforce the exact 73-column schema (asserts inside the module function)
if walk_ok:
    walk_final = pm.enforce_final_schema(walk_frame)
    print(f"Final shape: {walk_final.shape}")
    print("Positions 0-5  (metadata):", list(walk_final.columns[:6]))
    print("Positions 6-71 (features): 66 columns,", walk_final.columns[6], "...", walk_final.columns[71])
    print("Position 72    (target)  :", walk_final.columns[72])
    display(walk_final.head())
    display(walk_final[pm.LABEL_COLUMN].value_counts().to_frame("rows"))
    record("5.11 Final schema", "DONE", f"{INSPECT_FILE}: {walk_final.shape[0]:,} x {walk_final.shape[1]} (in memory, not saved)")
else:
    record("5.11 Final schema", "SKIPPED", "previous step did not run")

## 6. Output generation
Runs the repository's entry point **`pm.main()`**, the same code as running from a command prompt:
```
python preprocessing\process_mscad_73cols.py --base-dir D:\mscad --reference <MSCAD.csv> --imputation zero
```
It processes all six captures independently and writes into `<BASE_DIR>\CSV_73Cols\`:
* the six 73-column CSVs, keeping their original file names,
* `processing_summary.csv`, `class_counts_per_file.csv`, `aggregated_class_verification.csv`,
  `invalid_values_report.csv`, `schema_report.csv`, `validation_checks.csv`, `processing_report.md`,
  `run_metadata.json`, `processing_log.txt`,
* `_provenance\` (label source of every row) and `_diagnostics\` (ambiguous or rejected signatures).

Input files and `MSCAD.csv` are never modified (checked by SHA-256). Exit codes: **0** all checks passed ·
**1** a file failed or a validation check failed · **2** inputs missing · **3** reference missing or invalid
(stopped before writing).

In [ ]:
# [needs captures] [needs reference] 6.1 Run the full pipeline with pm.main()
EXIT_MEANING = {
    0: "all six files processed and all validation checks passed",
    1: "at least one file failed or a validation check FAILED (see Section 7)",
    2: "input directory or input files missing",
    3: "reference dataset missing or invalid: stopped before writing outputs",
}
PIPELINE_EXIT_CODE = None
if not RUN_FULL_PIPELINE:
    record("6.1 Run pipeline", "SKIPPED", "RUN_FULL_PIPELINE is False")
elif not (INPUTS_AVAILABLE and REFERENCE_AVAILABLE):
    record("6.1 Run pipeline", "SKIPPED",
           f"needs the six captures ({'found' if INPUTS_AVAILABLE else 'missing'}) "
           f"and MSCAD.csv ({'found' if REFERENCE_AVAILABLE else 'missing'})")
else:
    pipeline_argv = ["--base-dir", str(BASE_DIR), "--input-dir", str(INPUT_DIR), "--output-dir", str(OUTPUT_DIR),
                     "--reference", str(REFERENCE_PATH), "--imputation", IMPUTATION]
    print("pm.main(" + repr(pipeline_argv) + ")\n")
    configure_logging()
    t0 = time.time()
    try:
        PIPELINE_EXIT_CODE = pm.main(pipeline_argv)
    except Exception as exc:
        record("6.1 Run pipeline", "FAILED", f"{type(exc).__name__}: {exc}")
        raise
    finally:
        configure_logging()  # closes the log-file handler opened by pm.main()
    message = f"exit code {PIPELINE_EXIT_CODE} ({EXIT_MEANING.get(PIPELINE_EXIT_CODE, 'unknown')}) in {time.time() - t0:.1f}s"
    print("\n" + message)
    record("6.1 Run pipeline", "DONE" if PIPELINE_EXIT_CODE == 0 else "FAILED", message)

In [ ]:
# [needs captures] 6.2 Generated files and the dimensions of the six outputs (each reloaded separately)
OUTPUTS_WRITTEN = PIPELINE_EXIT_CODE in (0, 1)
if OUTPUTS_WRITTEN:
    generated = [
        {"File": p.relative_to(OUTPUT_DIR).as_posix(), "Size (KB)": round(p.stat().st_size / 1024, 1)}
        for p in sorted(OUTPUT_DIR.rglob("*")) if p.is_file()
    ]
    display(pd.DataFrame(generated))
    dim_rows = []
    for fname in pm.INPUT_FILES:
        path = OUTPUT_DIR / fname
        if not path.is_file():
            dim_rows.append({"Output": fname, "Rows": None, "Columns": None, "Schema OK": False, "Labels": "NOT WRITTEN"})
            continue
        out = pd.read_csv(path, dtype=str, keep_default_na=False, encoding="utf-8")
        dim_rows.append({
            "Output": fname,
            "Rows": len(out),
            "Columns": out.shape[1],
            "Schema OK": list(out.columns) == pm.FINAL_COLUMNS,
            "Labels": dict(out[pm.LABEL_COLUMN].value_counts()),
        })
        del out
    OUTPUT_DIMENSIONS = pd.DataFrame(dim_rows)
    display(OUTPUT_DIMENSIONS)
    record("6.2 Generated files", "DONE", f"{len(generated)} files in {OUTPUT_DIR}")
else:
    record("6.2 Generated files", "SKIPPED", "pipeline did not write outputs in this session")

In [ ]:
# [needs captures] 6.3 Verification reports written by the pipeline
if OUTPUTS_WRITTEN:
    display(Markdown("**processing_summary.csv** (one column per capture)"))
    display(pd.read_csv(OUTPUT_DIR / "processing_summary.csv").set_index("Filename").T)

    display(Markdown("**class_counts_per_file.csv**: rows per class, split by label source"))
    class_counts = pd.read_csv(OUTPUT_DIR / "class_counts_per_file.csv")
    display(class_counts[class_counts["Count"] > 0])

    display(Markdown("**aggregated_class_verification.csv**: totals across the six files vs Kaggle"))
    aggregated = pd.read_csv(OUTPUT_DIR / "aggregated_class_verification.csv")
    display(aggregated.drop(columns=["Explanation"]))
    for _, row in aggregated.dropna(subset=["Explanation"]).iterrows():
        print(f"- {row['Class']}: {row['Explanation']}")

    display(Markdown("**invalid_values_report.csv**"))
    invalid_report = pd.read_csv(OUTPUT_DIR / "invalid_values_report.csv")
    display(invalid_report if not invalid_report.empty else Markdown("_no invalid numeric values_"))

    display(Markdown("**schema_report.csv**"))
    display(pd.read_csv(OUTPUT_DIR / "schema_report.csv"))

    flagged = sorted((OUTPUT_DIR / pm.DIAGNOSTICS_DIR).glob("*.csv"))
    print("Flagged-signature files (_diagnostics):", [p.name for p in flagged] or "none")
    record("6.3 Verification reports", "DONE", "summary, class counts, aggregation, invalid values, schema")
else:
    record("6.3 Verification reports", "SKIPPED", "pipeline did not write outputs in this session")

In [ ]:
# [needs captures] 6.4 Label provenance of INSPECT_FILE, and agreement between the Section 5 walkthrough and the written file
if OUTPUTS_WRITTEN and (OUTPUT_DIR / INSPECT_FILE).is_file():
    provenance = pd.read_csv(OUTPUT_DIR / pm.PROVENANCE_DIR / INSPECT_FILE.replace(".csv", ".provenance.csv"))
    display(Markdown(f"**Label source per class in {INSPECT_FILE}** (`capture_rule` = single-class capture; "
                     "`kaggle_exact_66_features` / `kaggle_signature_12` = verified W-B-02 matches)"))
    display(pd.crosstab(provenance["Label"], provenance["Label_Source"], margins=True))

    if walk_final is not None:
        written = pd.read_csv(OUTPUT_DIR / INSPECT_FILE, dtype=str, keep_default_na=False, encoding="utf-8")
        consistency = {
            "same number of rows": len(written) == len(walk_final),
            "same column order": list(written.columns) == list(walk_final.columns),
            "same labels in same order": len(written) == len(walk_final)
                and bool((written[pm.LABEL_COLUMN].to_numpy() == walk_final[pm.LABEL_COLUMN].to_numpy()).all()),
            "same metadata/timestamps in same order": len(written) == len(walk_final)
                and bool((written[pm.METADATA_COLUMNS].to_numpy() == walk_final[pm.METADATA_COLUMNS].astype(str).to_numpy()).all()),
        }
        display(pd.Series(consistency, name="OK").to_frame())
        record("6.4 Walkthrough vs written output", "DONE" if all(consistency.values()) else "WARNING",
               "identical" if all(consistency.values()) else "differences: see table")
    else:
        record("6.4 Walkthrough vs written output", "SKIPPED", "Section 5 walkthrough did not run")
else:
    record("6.4 Walkthrough vs written output", "SKIPPED", "pipeline did not write outputs in this session")

## 7. Testing and validation
* **7.1** shows the checks `pm.main()` ran after reloading its own outputs from disk: schema, labels, sort
  order, metadata preserved, valid values unchanged, no duplicated or fabricated rows, inputs and reference unmodified.
* **7.2** runs the repository test suite `tests/test_process_mscad_73cols.py` in a separate Python process.
  It builds a **synthetic** dataset in a temporary folder (no real data needed and nothing in `D:\mscad` is
  touched) and checks exact expected counts. It uses `pytest` if installed, otherwise plain Python.
  Failures are reported, not skipped.

In [ ]:
# [needs captures] 7.1 Validation checks performed by the pipeline on its written outputs
if OUTPUTS_WRITTEN and (OUTPUT_DIR / "validation_checks.csv").is_file():
    checks = pd.read_csv(OUTPUT_DIR / "validation_checks.csv")
    VALIDATION_SUMMARY = checks["Status"].value_counts().to_dict()
    display(checks["Status"].value_counts().to_frame("checks"))
    not_passed = checks[checks["Status"] != "PASS"]
    if not_passed.empty:
        print("All checks passed.")
    else:
        display(Markdown("**Checks that did not pass:**"))
        display(not_passed)
    display(checks.pivot_table(index="Check", columns="File", values="Status", aggfunc="first").fillna(""))
    validation_status = ("FAILED" if VALIDATION_SUMMARY.get("FAIL", 0)
                         else "WARNING" if VALIDATION_SUMMARY.get("WARN", 0) else "DONE")
    record("7.1 Pipeline validation", validation_status,
           ", ".join(f"{k} {v}" for k, v in VALIDATION_SUMMARY.items()))
else:
    record("7.1 Pipeline validation", "SKIPPED", "pipeline did not write outputs in this session")

In [ ]:
# [no data] 7.2 Run the repository test suite (synthetic data, separate process)
TEST_RESULT = None
if not RUN_TEST_SUITE:
    record("7.2 Test suite", "SKIPPED", "RUN_TEST_SUITE is False")
elif not TEST_FILE.is_file():
    record("7.2 Test suite", "FAILED", f"test file not found: {TEST_FILE}")
else:
    use_pytest = importlib.util.find_spec("pytest") is not None
    test_cmd = ([sys.executable, "-m", "pytest", "-q", str(TEST_FILE)] if use_pytest
                else [sys.executable, str(TEST_FILE)])
    print("Running:", " ".join(test_cmd), "\n(this takes about 30-60 seconds)\n")
    t0 = time.time()
    env = dict(os.environ, PYTHONIOENCODING="utf-8")
    try:
        proc = subprocess.run(test_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True,
                              encoding="utf-8", errors="replace", timeout=1800, env=env)
        test_output = (proc.stdout or "") + (proc.stderr or "")
        returncode = proc.returncode
    except subprocess.TimeoutExpired as exc:
        test_output, returncode = f"TIMEOUT after {exc.timeout}s", -1
    elapsed = time.time() - t0
    lines = test_output.splitlines()
    result_lines = [l for l in lines if l.startswith(("PASS ", "FAILED", "ERROR")) or " passed" in l or " failed" in l
                    or "Error" in l or "assert" in l]
    print("Result lines:\n  " + "\n  ".join(result_lines or ["(none)"]))
    print("\nLast 25 lines of test output:\n" + "\n".join(lines[-25:]))
    TEST_RESULT = {"command": " ".join(test_cmd), "return code": returncode, "seconds": round(elapsed, 1),
                   "result lines": result_lines}
    if returncode == 0:
        record("7.2 Test suite", "DONE", f"passed (return code 0) in {elapsed:.0f}s")
    else:
        print("\n" + "!" * 70 + f"\nTEST SUITE FAILED (return code {returncode}). See the output above.\n" + "!" * 70)
        record("7.2 Test suite", "FAILED", f"return code {returncode}")

## 8. Final results
Built only from what ran in this session (`STEP_LOG` and the variables set above). Steps that did not run
are listed with the reason.

In [ ]:
# [no data] 8.1 Summary of this session
display(Markdown("### Steps"))
step_table = pd.DataFrame([{"Step": k, "Status": s, "Detail": d} for k, (s, d) in STEP_LOG.items()])
display(step_table)
display(step_table["Status"].value_counts().to_frame("steps"))

display(Markdown("### Dataset dimensions"))
if reference is not None:
    print(f"Kaggle reference: {reference.shape[0]:,} rows x {reference.shape[1]} columns ({REFERENCE_PATH})")
else:
    print("Kaggle reference: not loaded in this session")
if INPUT_DIMENSIONS is not None:
    display(INPUT_DIMENSIONS[["Capture", "Rows", "Columns"]])
else:
    print("Input captures: not loaded in this session")

display(Markdown("### Generated outputs"))
if OUTPUT_DIMENSIONS is not None:
    print(f"Output directory: {OUTPUT_DIR}  (pipeline exit code {PIPELINE_EXIT_CODE}: {EXIT_MEANING.get(PIPELINE_EXIT_CODE)})")
    display(OUTPUT_DIMENSIONS)
else:
    print("No outputs were generated in this session.")

display(Markdown("### Validation and tests"))
print("Pipeline validation checks:", VALIDATION_SUMMARY if VALIDATION_SUMMARY is not None else "not run in this session")
print("Test suite:", TEST_RESULT if TEST_RESULT is not None else "not run in this session")

display(Markdown("### Warnings, failures and skipped steps"))
issues = step_table[step_table["Status"] != "DONE"]
if issues.empty:
    print("None.")
else:
    display(issues)

### Interpreting the labels
* Single-class captures carry `capture_rule` labels. W-B-02 labels are `kaggle_exact_66_features`,
  `kaggle_signature_12` (matched Kaggle rows with exactly one permitted label) or `attack_session_rule`.
* `capture_reconciliation.csv` compares each capture with the Kaggle count of its class, and
  `aggregated_class_verification.csv` does it class by class: Normal 28,481 - Brute_Force 24 = 28,457,
  each class counted once. These are documented coverage/extraction variations, not labelling errors; any other
  difference fails validation.

* `extraction_artefacts.csv` counts rows flagged as CICFlowMeter artefacts (kept unchanged, see `_provenance/`).

### Known limitations of the existing code (documented, not changed)
* `pm._setup_logging()` adds a new log-file handler each time `pm.main()` is called, so calling it twice in
  one Python process duplicates log lines. This notebook closes old handlers before and after each run
  (`configure_logging()`).
* `pm._count_data_rows()` (reference search) reads with UTF-8 only. A candidate CSV with a valid header but
  non-UTF-8 bytes further down would raise `UnicodeDecodeError` outside the error handling in `pm.main()`.
  Setting `REFERENCE_OVERRIDE` avoids the search.

### Next steps
Model training (AE, CNN-LSTM, BiLSTM) and sequential pattern mining are not in the repository yet. When they
are added, the six files in `CSV_73Cols\` and the encoding `pm.LABEL_ENCODING` are their inputs.